# Batch Inference Drift Monitoring

Assumption:
- Assume that Titanic test data is your daily serving data (In reality, your daily serving data would be changing each day)

What we did previously:
- In our batch deployment, we have set a scheduled job that runs daily that predicts a daily serving table and save it to `ml_catalog.titanic_schema.titanic_prediction`. In this next, section we'll take the prediction table and build a dashboard to monitor model drift and feature drift

Goal:
- Everyday, we want to monitor feature drift and model performance and push the data into a dashboard

In [0]:
# Imports
from datetime import datetime
import warnings

import mlflow
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

warnings.filterwarnings("ignore")
mlflow.set_registry_uri("databricks-uc") # Set registry on top to set this aside

In [0]:
CATALOG_NAME = "ml_catalog"
SCHEMA_NAME = "titanic_schema"

# Load serving raw features & train raw features (To calculate feature drift)
TRAIN_RAW_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.train"
SERVING_RAW_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.test"

train_raw_df = spark.table(TRAIN_RAW_TABLE).toPandas()
serving_raw_df = spark.table(SERVING_RAW_TABLE).toPandas()

# Load serving prediction data & serving label (To calculate model drift)
SERVING_PREDICTION_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.titanic_prediction"
SERVING_LABEL_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.gender_submission"

serving_pred_df = spark.table(SERVING_PREDICTION_TABLE).toPandas()
serving_label_df = spark.table(SERVING_LABEL_TABLE).toPandas()

### Note on Label on Serving Data

If you are able to get label for your serving data, then we can calculate model performance on serving data.
- I.e. For fraud detection, after X days, you are able to get result back whether a transaction is fraud or not.  

However, not all prediction task, you can get label for your prediction no matter how long it takes. For example, if you're trying to predict someone's income, you can't get label for your prediction no matter how long you waited because it is a proprietary information. 

In our case, we assume that we have the label for our serving data, so we can calculate our model in production environment

# Measuring Feature Drift

In this section we will do these steps:
1. Identify the raw continuous and categorical input features used by the model
2. Compare the feature distribution of our latest serving data with the data used to train the model
3. Save daily feature drift metrics so we can show them in our monitoring dashboard and alerts

For continuous features, we will use Population Stability Index (PSI). For categorical features, we will compare the category distribution directly rather than measuring drift on the one-hot encoded columns individually.

Note: We measure feature drift on the raw features not after post-feature engineering because feature engineering already impute and modify the data which can affect the real distribution

Use this boilerplate code to measure feature drift

In [0]:
EPSILON = 1e-6

def alert_from_score(score, moderate=0.10, high=0.25):
    if pd.isna(score):
        return "unknown"
    if score >= high:
        return "high"
    if score >= moderate:
        return "moderate"
    return "low"


# Population Stability Index - Measures how much the population shifted between two numeric distributions
def compute_psi(train_series, serving_series, bins=10):
    ref = pd.to_numeric(train_series, errors="coerce").dropna()
    cur = pd.to_numeric(serving_series, errors="coerce").dropna()

    if ref.empty or cur.empty:
        return np.nan

    quantiles = np.linspace(0, 1, bins + 1)
    bin_edges = np.unique(ref.quantile(quantiles).values)

    if len(bin_edges) < 3:
        lower = min(ref.min(), cur.min())
        upper = max(ref.max(), cur.max())
        if lower == upper:
            return 0.0
        bin_edges = np.linspace(lower, upper, 4)

    ref_bins = pd.cut(ref, bins=bin_edges, include_lowest=True, duplicates="drop")
    cur_bins = pd.cut(cur, bins=bin_edges, include_lowest=True, duplicates="drop")

    ref_dist = ref_bins.value_counts(normalize=True, sort=False)
    cur_dist = cur_bins.value_counts(normalize=True, sort=False).reindex(ref_dist.index, fill_value=0.0)

    ref_values = np.clip(ref_dist.to_numpy(dtype=float), EPSILON, None)
    cur_values = np.clip(cur_dist.to_numpy(dtype=float), EPSILON, None)
    return float(np.sum((cur_values - ref_values) * np.log(cur_values / ref_values)))


# Jensen-Shannon divergence - Measures drift for a raw categorical feature by comparing category proportions
def compute_categorical_drift(train_series, serving_series):
    ref = train_series.astype("string").fillna("__missing__")
    cur = serving_series.astype("string").fillna("__missing__")

    categories = sorted(set(ref.unique()).union(set(cur.unique())))

    ref_dist = ref.value_counts(normalize=True).reindex(categories, fill_value=0.0)
    cur_dist = cur.value_counts(normalize=True).reindex(categories, fill_value=0.0)

    ref_values = np.clip(ref_dist.to_numpy(dtype=float), EPSILON, None)
    cur_values = np.clip(cur_dist.to_numpy(dtype=float), EPSILON, None)
    ref_values = ref_values / ref_values.sum()
    cur_values = cur_values / cur_values.sum()
    midpoint = 0.5 * (ref_values + cur_values)

    js_divergence = float(
        0.5 * (
            np.sum(ref_values * np.log(ref_values / midpoint))
            + np.sum(cur_values * np.log(cur_values / midpoint))
        )
    )

    return js_divergence

In [0]:
BATCH_DATE = datetime.now().strftime("%Y-%m-%d")

FEATURE_DRIFT_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.feature_drift_daily"

# Best Practice: This should be stored in a config.py file
continuous_features = ["Fare", "Age", "Pclass", "SibSp", "Parch"]
categorical_features = ["Sex"]

feature_drift_records = []

for feature_name in continuous_features:
    train_feat = train_raw_df[feature_name]
    serving_feat = serving_raw_df[feature_name]
    drift_score = compute_psi(train_feat, serving_feat)

    feature_drift_records.append(
        {
            "batch_date": BATCH_DATE,
            "feature_name": feature_name,
            "feature_type": "continuous",
            "drift_metric": "population_stability_index",
            "drift_score": drift_score,
            "alert_level": alert_from_score(drift_score),
            "train_mean": float(pd.to_numeric(train_feat, errors="coerce").mean()),
            "serving_mean": float(pd.to_numeric(serving_feat, errors="coerce").mean()),
            "train_null_rate": float(train_feat.isna().mean()),
            "serving_null_rate": float(serving_feat.isna().mean())
        }
    )

for feature_name in categorical_features:
    train_feat = train_raw_df[feature_name]
    serving_feat = serving_raw_df[feature_name]
    drift_score = compute_categorical_drift(train_feat, serving_feat)

    train_dist = train_feat.astype("string").fillna("__missing__").value_counts(normalize=True)
    serving_dist = serving_feat.astype("string").fillna("__missing__").value_counts(normalize=True)
    categories = sorted(set(train_dist.index).union(set(serving_dist.index)))
    train_dist = train_dist.reindex(categories, fill_value=0.0)
    serving_dist = serving_dist.reindex(categories, fill_value=0.0)

    feature_drift_records.append(
        {
            "batch_date": BATCH_DATE,
            "feature_name": feature_name,
            "feature_type": "categorical",
            "drift_metric": "jensen_shannon_divergence",
            "drift_score": drift_score,
            "alert_level": alert_from_score(drift_score),
            "train_mean": None,
            "serving_mean": None,
            "train_null_rate": float(train_feat.isna().mean()),
            "serving_null_rate": float(serving_feat.isna().mean())
        }
    )

feature_drift_df = pd.DataFrame(feature_drift_records)
feature_drift_df['batch_date'] = pd.to_datetime(feature_drift_df['batch_date'])
spark.createDataFrame(feature_drift_df).write.mode("append").saveAsTable(FEATURE_DRIFT_TABLE)

print(f"Continuous features monitored with PSI: {continuous_features}")
print(f"Categorical features monitored with Jensen-Shannon divergence: {categorical_features}")
print(f"Saved feature drift metrics to {FEATURE_DRIFT_TABLE}")
display(feature_drift_df.head(10))


# Measuring Model Drift

Feature drift does not always mean model performance will degrade. A model may still generalize well even when the underlying feature distribution changes. That is why we also monitor model drift, so we can determine whether the drift is actually affecting prediction quality and whether action is needed.

In this section we will:
1. Compare model performance on training data versus the latest serving batch
2. Calculate daily model drift metrics and save them to a monitoring table
3. Use model drift together with feature drift to decide when to investigate or retrain

How to use model drift and feature drift
- If a feature shows significant drift, check model performance. If performance is still acceptable, no immediate action may be needed.
- If model performance degrades, review feature drift to help determine whether the degradation is caused by changes in the underlying data distribution or other factors

Use this boilerplate code to measure model drift

In [0]:
def safe_classification_metric(metric_name, y_true, y_pred):
    if metric_name == "accuracy":
        return float(accuracy_score(y_true, y_pred))
    if metric_name == "precision":
        return float(precision_score(y_true, y_pred, zero_division=0))
    if metric_name == "recall":
        return float(recall_score(y_true, y_pred, zero_division=0))
    if metric_name == "f1_score":
        return float(f1_score(y_true, y_pred, zero_division=0))
    if metric_name == "auc":
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(roc_auc_score(y_true, y_pred))
    raise ValueError(f"Unsupported metric: {metric_name}")

In [0]:
# Load the metrics we got from train/eval 
from mlflow.tracking import MlflowClient

model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.example_titanic_model"
model_alias = "production"

client = MlflowClient()
model_version = client.get_model_version_by_alias(model_name, model_alias)
run_id = model_version.run_id
run = client.get_run(run_id)

train_metrics = run.data.metrics
print(f"Loaded metrics for {model_name}@{model_alias} (run_id={run_id})")
display(pd.DataFrame([train_metrics]))


In [0]:
# Load our serving prediction data and join to its label
serving_pred_with_label = pd.merge(serving_pred_df, serving_label_df, on=['PassengerId'])
serving_pred_with_label

In [0]:
# Calculate Model Performance Drift

BATCH_DATE = datetime.now().strftime("%Y-%m-%d")
model_drift_records = []

model_metric_thresholds = {
    "accuracy": 0.05,
    "f1_score": 0.05,
    "auc": 0.05,
    "prediction_positive_rate": 0.10,
    "label_positive_rate": 0.10,
    "prediction_distribution_drift": 0.10,
}

for metric_name in ["accuracy", "auc", "f1_score"]:
    train_metric = train_metrics[f"eval_{metric_name}"] # Simply retrieve the eval metric from our train/eval run
    serving_metric = safe_classification_metric(metric_name, serving_pred_with_label["Survived"], serving_pred_with_label["prediction"]) # calculate the metric for serving run
    drift_score = max(train_metric - serving_metric, 0) 

    model_drift_records.append(
        {
            "batch_date": BATCH_DATE,
            "metric_name": metric_name,
            "train_value": train_metric,
            "serving_value": serving_metric,
            "delta": serving_metric - train_metric,
            "drift_score": drift_score,
            "alert_level": alert_from_score(
                drift_score,
                moderate=model_metric_thresholds[metric_name],
                high=model_metric_thresholds[metric_name] * 2,
            ),
        }
    )



model_drift_df = pd.DataFrame(model_drift_records)
MODEL_DRIFT_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.model_drift_daily"
model_drift_df['batch_date'] = pd.to_datetime(model_drift_df['batch_date'])
spark.createDataFrame(model_drift_df).write.mode("append").saveAsTable(MODEL_DRIFT_TABLE)

print(f"Saved model drift metrics to {MODEL_DRIFT_TABLE}")
display(model_drift_df)


# Creating Alerts

In this section we will:
1. Combine moderate and high feature drift and model drift signals into a single alerts dataset
2. Save alert records so they can be shown in dashboards and reviewed by stakeholders
3. Prepare this notebook to run as a scheduled daily job that continuously monitors drift and creates alerts



In [0]:
feature_alerts = feature_drift_df.loc[
    feature_drift_df["alert_level"].isin(["moderate", "high"]),
    ["batch_date", "feature_name", "drift_metric", "drift_score", "alert_level"],
].copy()

feature_alerts["alert_type"] = "feature_drift"
feature_alerts["alert_message"] = feature_alerts.apply(
    lambda row: f"{row['feature_name']} drift score reached {row['drift_score']:.3f} using {row['drift_metric']}",
    axis=1,
)
feature_alerts = feature_alerts.rename(columns={"feature_name": "entity_name", "drift_metric": "metric_name", "drift_score": "metric_value"})

In [0]:
feature_drift_df


In [0]:
feature_alerts


NO alerts are logged because feature drift are still within expected bounds

In [0]:
model_alerts = model_drift_df.loc[
    model_drift_df["alert_level"].isin(["moderate", "high"]),
    ["batch_date", "metric_name", "drift_score", "alert_level", "serving_value", "train_value"],
].copy()
model_alerts["alert_type"] = "model_drift"
model_alerts["alert_message"] = model_alerts.apply(
    lambda row: f"{row['metric_name']} moved from {row['train_value']:.3f} to {row['serving_value']:.3f}",
    axis=1,
)
model_alerts = model_alerts.rename(columns={"metric_name": "entity_name", "drift_score": "metric_value"})
model_alerts["metric_name"] = "metric_drift"
model_alerts = model_alerts[["batch_date", "entity_name", "metric_name", "metric_value", "alert_level", "alert_type", "alert_message"]]

In [0]:
model_drift_df

In [0]:
model_alerts


Same as well with model_drift. If serving performance is higher than train/eval performance then we don't output an alert. Only when it is lower then we output an alert.

In [0]:
alerts_df = pd.concat(
    [
        feature_alerts,
        model_alerts,
    ],
    ignore_index=True,
)

if alerts_df.empty:
    alerts_df = pd.DataFrame(
        [
            {
                "batch_date": BATCH_DATE,
                "entity_name": "batch_summary",
                "metric_name": "no_alerts",
                "metric_value": 0.0,
                "alert_level": "low",
                "alert_type": "summary",
                "alert_message": "No medium or high drift detected for this batch",
            }
        ]
    )

ALERT_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.drift_alerts_daily"
alerts_df['batch_date'] = pd.to_datetime(alerts_df['batch_date'])
spark.createDataFrame(alerts_df).write.mode("append").saveAsTable(ALERT_TABLE)

print(f"Saved alerts to {ALERT_TABLE}")
display(alerts_df)


# Set this notebook as a scheduled daily job

Using the same method in lab 2c, we set a daily scheduled run for this notebook.
Optimally, you want to set this notebook to run once your daily serving label is able to be captured. I.e. For Fraud, let's say everyday at 8am, Data Engineers store the list of fraudulent transaction in a table. If so, you run this query at 9am everyday once that label already exist so you can check how well your model prediction performs compared to actual frauds.


# Setting up a Dashboard & Alert

Next, we will set up a dashboard that is connected to the feature_drift_daily table and model_drift_daily table to monitor feature drift and model performance daily. And then we will set up an alert that is connected to the drift_alerts_daily table to send an email whenever there's a moderate or high alerts